In [ ]:
# =============================================================================
# Q-LEARNING, SARSA, DQN — learn WHILE you walk (not only at the finish line)
# =============================================================================
#
# Last notebook (RL foundations):
#   Dynamic Programming → needs the full rulebook P (model-based planning)
#   Monte Carlo         → learns from COMPLETE episodes (wait until the end)
#
# This notebook: Temporal Difference (TD) methods.
#   Update your guess AFTER EACH STEP, using the reward you just got PLUS
#   your current guess of the next state's value. No rulebook. No waiting
#   for the episode to finish.
#
# Analogy — grading a hike as you go:
#   Monte Carlo: finish the trail, then score every mile from total time.
#   TD:         at each milepost, update "how good is this spot?" using
#               (time so far) + (my estimate of the remaining trail).
#
# That "estimate of the remaining trail" is bootstrapping — learning from
# your own current estimates. Powerful, a bit like standing on your own
# shoulders. Works online, step by step.
#


# -----------------------------------------------------------------------------
# 1. TEMPORAL DIFFERENCE (TD) LEARNING — the big idea
# -----------------------------------------------------------------------------
#
# Recall: return from time t is the discounted sum of future rewards:
#   G_t = R_{t+1} + γ R_{t+2} + γ² R_{t+3} + …
#
# Monte Carlo waits for the whole G_t, then averages.
#
# TD(0) for state values instead uses a ONE-STEP target:
#   target ≈ R_{t+1} + γ V(S_{t+1})     ← "reward now + guess of what comes next"
#
# Update:
#   V(S_t) ← V(S_t) + α [ target − V(S_t) ]
#
# The thing in brackets is the TD error δ:
#   δ = (what just happened + my guess for next) − (what I thought before)
#
# If δ > 0: "that step was BETTER than I expected → raise V(S_t)"
# If δ < 0: "worse than expected → lower V(S_t)"
#
# α (learning rate): how big a step toward the new target (e.g. 0.1).
# γ (discount): how much future rewards matter (same as before).
#
# For CONTROL (picking actions) we track Q(s,a) = "how good is action a in s?"
# Two famous recipes: SARSA and Q-learning. They differ in ONE word of the
# target — and that one word changes their personality.
#


# -----------------------------------------------------------------------------
# 2. SARSA — On-policy TD control  (State → Action → Reward → State → Action)
# -----------------------------------------------------------------------------
#
# Name comes from the tuple it uses each update:
#   (S, A, R, S', A')
#
# After taking A in S, seeing R and landing in S', SARSA ALSO picks the NEXT
# action A' the SAME way it usually acts (e.g. ε-greedy), then updates:
#
#   Q(S,A) ← Q(S,A) + α [ R + γ Q(S', A') − Q(S,A) ]
#                              └─────┬─────┘
#                         value of the action we WILL take next
#
# On-policy = "learn about the policy you are actually following."
# Including the exploration! If ε-greedy sometimes walks off a cliff,
# SARSA learns "near the cliff, that Q is dangerous" because A' might be
# the clumsy exploratory step.
#
# Personality: cautious. Good when exploration is risky in the real world
# (robots, medicine). The learned policy matches "how I behave while learning."
#


# -----------------------------------------------------------------------------
# 3. Q-LEARNING — Off-policy TD control  (dream of the best next move)
# -----------------------------------------------------------------------------
#
# Same setup: take A in S (often still ε-greedy for exploration), see R, S'.
# But the TARGET pretends the NEXT action is the BEST one, not the one you
# might actually take:
#
#   Q(S,A) ← Q(S,A) + α [ R + γ max_{a'} Q(S', a') − Q(S,A) ]
#                              └──────────┬──────────┘
#                         value of the BEST action available next
#
# Off-policy = "learn about a DIFFERENT (greedy) policy while behaving with
# exploration." Behavior policy explores; target policy is greedy.
#
# Personality: optimistic / bold. Learns the optimal path even if while
# training you sometimes take dumb exploratory moves. Classic cliff-walk
# demo: Q-learning hugs the cliff edge (shortest path); SARSA stays safer
# inland (because it "fears" its own ε-slips).
#
# Tiny comparison table:
#
#   Method       Target uses              Learns about          Typical vibe
#   -----------  -----------------------  --------------------  -------------
#   SARSA        Q(S', A') you will take  the exploring policy  cautious
#   Q-learning   max_a Q(S', a')          the greedy optimal    ambitious
#


# -----------------------------------------------------------------------------
# 4. DEEP Q-NETWORK (DQN) — Q-learning when the table does not fit
# -----------------------------------------------------------------------------
#
# Tabular Q: one number per (state, action). Fine for a 4×4 grid.
# Broken for Atari pixels or big continuous spaces — too many states.
#
# DQN idea: replace the table with a neural net Q(s, a; θ) that OUTPUTS
# action-values from a state (e.g. image → 4 joystick scores).
#
# Still the Q-learning target, but now a regression loss:
#   y = R + γ max_{a'} Q(S', a'; θ⁻)     (θ⁻ = a frozen "target network")
#   loss ≈ (y − Q(S, A; θ))²
#
# Two tricks that made deep RL actually work (Mnih et al., 2015):
#
#   (1) Experience replay
#       Store past transitions (S,A,R,S') in a big buffer. Train on RANDOM
#       mini-batches from the buffer — breaks the "correlated consecutive
#       frames" problem, like shuffling a dataset.
#
#   (2) Target network
#       Keep a slow-copy θ⁻ of the net for computing y. Update θ⁻ only
#       every N steps (or soft-update). Stops the moving-target chase
#       where the thing you chase is also the thing you train.
#
# Rough mental model:
#   Tabular Q-learning = flashcards for every room-door pair.
#   DQN               = a brain that looks at the room and guesses door scores.
#


# -----------------------------------------------------------------------------
# HOW THE PIECES FIT (roadmap for this notebook)
# -----------------------------------------------------------------------------
#
#   MC (previous)     wait for full return G          stable, slow, needs episodes
#   TD / SARSA        bootstrap with Q(S', A')        online, on-policy
#   Q-learning        bootstrap with max Q(S', ·)     online, off-policy
#   DQN               Q-learning + neural net         scales to big / pixel states
#
# Same goal as always: find a good policy π that maximizes discounted return.
# Different tools for when you get the learning signal (end vs each step) and
# how you represent Q (table vs network).
#
# Next cells: implement SARSA & Q-learning on GridWorld, compare them, then
# a small DQN sketch so the "table → network" jump feels concrete.
#
